# 01 - Build Span-Completion Instruction JSONL

Convert existing full-sequence RefSeq instruction rows into profile-conditioned protein span-completion rows.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_dir_for_import(start: Path) -> Path:
    candidates: list[Path] = []
    env_value = os.environ.get("MDNAC_REPO_DIR")
    if env_value:
        candidates.append(Path(env_value).expanduser())
    resolved_start = start.expanduser().resolve()
    candidates.extend([resolved_start, *resolved_start.parents])
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if (resolved / "pyproject.toml").exists() and (resolved / "libs").is_dir():
            return resolved
    raise RuntimeError("Could not locate MDNAC repo. Run inside the repo or set MDNAC_REPO_DIR.")


REPO_DIR = find_repo_dir_for_import(Path.cwd())
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from libs.notebook_runtime import bootstrap_notebook

RUNTIME = bootstrap_notebook(REPO_DIR)
REPO_DIR = Path(RUNTIME["repo_dir"])
print(json.dumps(RUNTIME, indent=2, ensure_ascii=False))

In [ ]:
SOURCE_JSONL = REPO_DIR / "data/compiled/refseq_bacteria_protein/instruction.jsonl"
OUTPUT_DIR = REPO_DIR / "data/compiled/refseq_bacteria_span_completion"
OUTPUT_JSONL = OUTPUT_DIR / "instruction.jsonl"
STATS_JSON = OUTPUT_DIR / "stats.json"

PARAMS = {
    "examples_per_sequence": 4,
    "min_sequence_length": 64,
    "max_sequence_length": 1024,
    "min_mask_length": 8,
    "max_mask_length": 64,
    "left_flank_size": 64,
    "right_flank_size": 64,
    "seed": 42,
    "mask_policies": ["random_span", "n_terminal_span", "c_terminal_span"],
}

assert SOURCE_JSONL.is_file(), f"Missing source instruction JSONL: {SOURCE_JSONL}"
print(json.dumps({"source": str(SOURCE_JSONL), "output": str(OUTPUT_JSONL), "stats": str(STATS_JSON), "params": PARAMS}, indent=2))

In [ ]:
from libs.protein_completion import convert_instruction_jsonl_to_span_jsonl, validate_jsonl_file

stats = convert_instruction_jsonl_to_span_jsonl(
    SOURCE_JSONL,
    OUTPUT_JSONL,
    stats_path=STATS_JSON,
    **PARAMS,
)
validation = validate_jsonl_file(OUTPUT_JSONL)

assert validation["valid_rows"] == stats["generated_span_rows"]
assert OUTPUT_JSONL.is_file()
assert STATS_JSON.is_file()

print("source rows:", stats["source_rows"])
print("accepted source rows:", stats["accepted_source_rows"])
print("skipped rows:", stats["skipped_rows"])
print("generated span rows:", stats["generated_span_rows"])
print("skip reasons:", json.dumps(stats["skip_reasons"], indent=2))
print("mask length distribution:", json.dumps(stats["mask_length_distribution"], indent=2))
print("validated rows:", validation["valid_rows"])
print("source-row immutability: checked during conversion")

In [ ]:
def clipped_payload(payload: dict, *, max_output_chars: int = 96) -> dict:
    clipped = dict(payload)
    output = clipped.get("output")
    if isinstance(output, str) and len(output) > max_output_chars:
        clipped["output"] = output[:max_output_chars] + f"...({len(output)} aa total)"
    return clipped


print("Before row:")
print(json.dumps(clipped_payload(stats["example_before"] or {}), indent=2, ensure_ascii=False))
print("\nAfter row:")
print(json.dumps(clipped_payload(stats["example_after"] or {}), indent=2, ensure_ascii=False))

In [ ]:
from libs.protein_completion import validate_span_completion_row

before = stats["example_before"]
after = stats["example_after"]
assert before is not None and after is not None, "No sample row was generated. Check skip_reasons in stats.json."
sample_validation = validate_span_completion_row(after, original_sequence=before["output"])

mask_length = sample_validation["mask_length"]
assert after["output"] == before["output"][sample_validation["mask_start"]:sample_validation["mask_end"]]
assert len(after["output"]) == mask_length
assert f"<MASK_{mask_length}>" in after["input"]
assert "<MASK_" in after["input"] and after["input"].count("<MASK_") == 1

print(json.dumps(sample_validation, indent=2))